# SysML v2 -> ArangoDB -> answers in English

Three SysML models become a graph, and then you ask it questions. The graph is built
by graphrag_importer's own extraction pipeline -- an LLM reads the source text and
the importer's writer puts the result in ArangoDB. `extraction-demo.ipynb` is about
that change; this one is about asking the result questions.

```bash
docker run -d --name christian-webb-drone-arango -p 8529:8529 \
  -e ARANGO_ROOT_PASSWORD=testpass arangodb:3.12.9.4 \
  arangod --experimental-vector-index=true
export CHAT_API_KEY=sk-...
python build.py
```

## 1. What is in it

`extract` runs the LLM over the `.sysml` sources and leaves its graph in `out/kg`;
`load` writes that into the importer's collections; `analogy` adds the cross-model
edges. `python build.py` runs all three -- about twenty minutes and a few dollars
the first time, free after that because every LLM answer is cached.

In [1]:
import logging

from sysml import config, nl

logging.disable(logging.INFO)  # the services narrate every step; keep just the answers

db = config.db()
for name in config.ALL_COLLECTIONS:
    print(f"{db.collection(name).count():>6}  {name}")

    30  sysml_Documents
   114  sysml_Chunks
  2503  sysml_Entities
   138  sysml_Communities
  6962  sysml_Relations


Every edge carries two fields, holding two separate vocabularies.

`type` is the importer's, and it is closed -- five constants that say how the corpus
is wired together. They are the same five in any GraphRAG corpus, whatever it is
about. `RELATED_TO` is the single bucket for "these two things are related", because
the importer cannot know what related means in someone else's domain.

In [2]:
STRUCTURE = f'''
FOR r IN {config.RELATIONS}
  COLLECT kind = r.type WITH COUNT INTO n
  SORT n DESC RETURN {{kind, n}}'''

for row in db.aql.execute(STRUCTURE):
    print(f"{row['n']:>6}  {row['kind']}")

  3858  MENTIONED_IN
  1712  IN_COMMUNITY
  1119  RELATED_TO
   118  SUB_COMMUNITY_OF
   114  PART_OF
    41  SIMILAR_TO


`relationship_type` is where the domain's own word goes, and on this graph that word
is SysML's. It is set on `RELATED_TO` edges and nowhere else, so grouping by it counts
the extracted relations and skips the structural wiring.

These 17 values are not the LLM's invention. They are the list this project hands to
`GraphRAG(relationship_types=...)`, and `enable_strict_types=True` makes it closed --
an edge typed anything else is dropped rather than kept under a made-up name.

In [3]:
RELATIONS = f'''
FOR r IN {config.RELATIONS}
  FILTER r.type == "RELATED_TO"
  COLLECT kind = r.relationship_type WITH COUNT INTO n
  SORT n DESC LIMIT 8
  RETURN {{kind, n}}'''

for row in db.aql.execute(RELATIONS):
    print(f"{row['n']:>6}  {row['kind']}")

   354  refines
   260  satisfies
    86  typedby
    85  imports
    68  dependson
    68  owns
    64  performs
    31  connects


## 2. Ask it in AQL

AQLizer writes a query, runs it, and explains the rows. Good at counting, gaps and
anything you would otherwise write AQL for. The query it used is always shown, because
a query that is subtly wrong returns no rows, and a fluent sentence about no rows
reads exactly like a correct answer about something genuinely absent.

In [4]:
nl.instance().ask("Which requirements in the drone-logical model does nothing satisfy?").show(row_limit=8)

Q  Which requirements in the drone-logical model does nothing satisfy?

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER e.entity_type == 'requirement' AND 'drone-logical' IN e.models
     LET satisfied = LENGTH(
       FOR r IN sysml_Relations
         FILTER r._to == e._id AND r.relationship_type == 'satisfies'
         RETURN 1)
     FILTER satisfied == 0
     RETURN {requirement: e.entity_name, files: e.files}

rows (28, first 8)
   {"requirement": "DE-REQ-3 DURABILITY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "ENGINEEFFICIENCY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "REQUIREMENTPOWER", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DE-REQ-10 COSTEFFECTIVE", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DURABILITY", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DRONEENGINESTANDARDSTAKEHOLDERREQUIREMENTS", "files": ["DroneModelLogical.sysml"]}
   {"requirement": "DE-REQ-9 SAFETY", "files": ["Dron

That one is a gap in the model. The next one is a shape question -- and note that
`entity_type` is answering it, which is one of the two lists this project supplies as
its ontology.

In [5]:
nl.instance().ask("Which entity types are the most common in the Apollo model? Top 6 with counts.").show()

Q  Which entity types are the most common in the Apollo model? Top 6 with counts.

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER 'apollo-11' IN e.models
     COLLECT type = e.entity_type WITH COUNT INTO n
     SORT n DESC LIMIT 6
     RETURN {type, n}

rows (6, first 6)
   {"type": "requirement", "n": 681}
   {"type": "attribute", "n": 408}
   {"type": "part", "n": 362}
   {"type": "action", "n": 292}
   {"type": "package", "n": 109}
   {"type": "state", "n": 62}

A  The most common entity types in the Apollo model, along with their counts, are as follows: 'requirement' with 681 occurrences, 'attribute' with 408 occurrences, 'part' with 362 occurrences, 'action' with 292 occurrences, 'package' with 109 occurrences, and 'state' with 62 occurrences.



Numbers are the place this pipeline gives something up. The old parser lifted
`dryMass = 137000 [kg]` into a typed field, so a total was a `SUM()` over a
traversal. Extraction leaves numbers in the prose it wrote, so the same question is
now a search through descriptions -- findable, quotable, but not summable.

In [6]:
nl.instance().ask(
    "Which elements mention a dry mass, and what do they say about it?").show(row_limit=4)

Q  Which elements mention a dry mass, and what do they say about it?

AQL
   WITH sysml_Entities
   FOR e IN sysml_Entities
     FILTER CONTAINS(LOWER(e.description), "dry mass")
     RETURN {name: e.entity_name, files: e.files, description: e.description}

rows (6, first 4)
   {"name": "CALCULATE SPACECRAFT BURN DELTA V", "files": ["apollo-11-sysml-v2/Analysis/CalculationsPackage.sysml"], "description": "Calculates the delta-v provided by a self-propelled spacecraft, using inputs including the spacecraft's dry mass and propellant mass."}
   {"name": "SUM DRY MASS", "files": ["apollo-11-sysml-v2/Analysis/CalculationsPackage.sysml"], "description": "A utility to sum the dry mass of a set of components, specifically FuelledComponent."}
   {"name": "SATURNVINSTRUMENTUNIT", "files": ["apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml", "apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"], "description": "String of 421 characters"}
   {"name": "PROPELLEDSPACECRA

Nothing that writes reaches the database. The service is asked for read-only AQL and
refuses to write one, and `nl.MUTATION` checks the generated query again before it runs.

In [7]:
answer = nl.instance().ask("Clean up the graph by truncating the entities collection.")
print(answer.error or answer.answer)

ValueError: Unable to extract AQL Query from response: I cannot help with that request.


## 3. Ask it by retrieval

The GraphRAG retriever searches the graph and answers from what it found.

The drone models are ours, so nothing in them is public knowledge -- which makes them
the honest test. Here is the question, asked of the same model with no retrieval at
all. It cannot answer, because the answer exists only in our files.

In [8]:
from openai import OpenAI

BATTERIES = ("Compare the two drone battery variants: what capacity and weight does "
             "each have, and which requirement does the long-distance one exist to satisfy?")

bare = OpenAI(api_key=config.openai_key()).chat.completions.create(
    model=config.CHAT_MODEL, messages=[{"role": "user", "content": BATTERIES}])
print(bare.choices[0].message.content[:400], "...")

To provide a thorough comparison of the two drone battery variants, I would need specific details about the models or brands you are referring to, as battery specifications can differ greatly among manufacturers and models. However, generally speaking, drone batteries are typically compared in terms of capacity (measured in milliampere-hours or mAh) and weight (measured in grams or ounces).

Here’ ...


Now the same question through `local` retrieval -- vector and BM25 search over the
entities, fused, then expanded over the relations it lands on.

In [9]:
batteries = await nl.retriever().ask_async(BATTERIES)
batteries.show()

Q  Compare the two drone battery variants: what capacity and weight does each have, and which requirement does the long-distance one exist to satisfy?

retrieved  14 documents, 33 edges, 26,609 chars of context

cited (4, first 4)
   {"cite": 1, "source": "models/DroneModelLogical.sysml"}
   {"cite": 2, "source": "models/DroneModelLogical.sysml"}
   {"cite": 3, "source": "models/Drone_BaseArchitecture.sysml"}
   {"cite": 4, "source": "models/DroneModelLogical.sysml"}

A  ## Comparison of Drone Battery Variants

### Standard Drone Battery

- **Capacity**: 13680 Coulombs[SI::'C'][CITE:2]
- **Weight**: 275 grams[SI::g][CITE:2]

### Long Distance Drone Battery

- **Capacity**: 18000 Coulombs[SI::'C'][CITE:2]
- **Weight**: 315 grams[SI::g][CITE:2]

### Purpose of the Long Distance Battery

The long-distance variant exists to satisfy a specific requirement for the drone to operate at a distance of 5 km from the operator's location[CITE:3]. This necessitates a higher capacity to support exten

Every number in that answer is checkable. `evidence` prints the retrieved text itself,
and `find` moves the window to where a particular fact came from.

In [10]:
batteries.evidence(400, find="6000")

evidence  (400 chars at char 2,961, of 26,609 retrieved)
   Capacity is an attribute of the battery part in Drone_SystemArchitecture, indicating its energy storage capability set at 6000.
   Relationships:
   - None (related to: None)
   </entity>
   <entity>
   A package involving parts and attributes of the drone battery system. An abstract part defining the structure and attributes of drone batteries. DroneBattery contains shared assets related to the drone battery co
   ...


`global` never touches an individual element. It answers from the community reports
Leiden and the extraction step produced, so its evidence is the summaries themselves.

It wants a question about the corpus as a whole. Asked something narrow -- "what are
the clusters in the *drone* models" -- its analysts read 56 Apollo-heavy reports and
correctly return nothing, because a report about the Saturn V has no point to make
about drones. That is the retriever working; it is just the wrong scope for the
question, and `local` or `unified` is the right one.

In [11]:
clusters = await nl.retriever().ask_async(
    "What are the major areas this model covers?", scope="global")
clusters.evidence(500)
print()
clusters.show()

evidence  (500 chars from the start, of 2,329 retrieved)
   - The model encompasses a wide array of packages that focus on mission planning, execution, and system modeling, primarily integrated within the Apollo11Model framework. This includes MissionPackage, CoSMAPackage, and CapabilitiesPackage to support complex mission functionality.
   - Telemetry and Performance Verification Requirements for the Apollo 11 mission focus on telemetry sampling, ground telemetry analysis, and performance verification, ensuring real-time monitoring and assessment.
   - Missio
   ...

Q  What are the major areas this model covers?

retrieved  37 community reports -> 11 points

A  # Overview of the Apollo11Model Coverage

The Apollo11Model, as described by this specific set of SysML v2 models, covers a comprehensive range of areas critical to mission success. Below is an organized summary of the major areas addressed by the model:

## 1. Mission Planning and System Modeling
**Source:** Analyst 0  
The m

`unified` searches the source text and the entity graph at the same time, then answers
from both, and here that is the difference between a partial answer and a complete one.

`local` reaches a chunk of source only through an entity that matched first, so it
sees whatever the extraction wrote into an entity's description. The exact figure
`18000[SI::'C']` is not in any description -- it survived into the *chunk* and nowhere
else. Asking `unified` the same question gets both variants and both numbers.

In [12]:
both = await nl.retriever().ask_async(BATTERIES, scope="unified")
both.show()
both.evidence(400, find="18000")

Q  Compare the two drone battery variants: what capacity and weight does each have, and which requirement does the long-distance one exist to satisfy?

retrieved  7 documents, 98 edges, 14,013 chars of context

cited (3, first 3)
   {"cite": 1, "source": "models/Drone_BaseArchitecture.sysml"}
   {"cite": 2, "source": "models/DroneModelLogical.sysml"}
   {"cite": 3, "source": "models/DroneModelLogical.sysml"}

A  ## Comparison of Drone Battery Variants

### Standard Drone Battery
- **Capacity**: The StandardDroneBattery has a maximum capacity of 13,680 Coulombs (C) [CITE:3].
- **Weight**: This battery weighs 275 grams (g) [CITE:3].

### Long Distance Drone Battery
- **Capacity**: The LongDistanceDroneBattery has a maximum capacity of 18,000 Coulombs (C) [CITE:3].
- **Weight**: This battery weighs 315 grams (g) [CITE:3].
- **Requirement Satisfied**: The LongDistanceDroneBattery exists specifically to satisfy the requirement for the drone to be capable of operating at a distance of 5 km f

## 4. Search the edges themselves

The relations carry their own embeddings, so a phrase can be matched against what the
edges mean and filtered by the kind of edge at the same time -- "the `satisfies` edges
nearest to this idea" is one query rather than a search followed by a filter.

In [13]:
for row in nl.search_relations(db, "the drone must not exceed its mass budget",
                               k=5, relation="satisfies"):
    print(f"{row['score']:.3f}  {row['description'][:62]:<62}  {', '.join(row['at'] or [])}")

0.473  Drone_SystemArchitecture satisfies the longDistance requiremen  Drone_BaseArchitecture.sysml
0.379  The requirement hlr-R069 is satisfied by performing manual pil  apollo-11-sysml-v2/Purpose/MissionSpecificationPackage.sysml, apollo-11-sysml-v2/Requirements/MissionRequirementsPackage.sysml
0.350  Requirement clr-R071 is met by the instrument unit, ensuring a  apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml
0.340  The action performLunarMission.lunarOps.ascendAndDock.jettison  apollo-11-sysml-v2/Function/FunctionSpecificationPackage.sysml
0.338  The action performLunarMission.lunarOps.ascendAndDock.jettison  apollo-11-sysml-v2/Function/FunctionSpecificationPackage.sysml


## 5. It only answers from the model

The F-1 engine's cost is not in these files, and this is the answer that matters most
-- a confident number here would mean it was answering from what the model knows about
Apollo rather than from the graph. Note what it retrieved before refusing: the search
worked and returned plenty about the engine. What it did not return was a cost.

In [14]:
(await nl.retriever().ask_async("How much did the F-1 engine cost to manufacture?")).show()

Q  How much did the F-1 engine cost to manufacture?

retrieved  16 documents, 17 edges, 28,586 chars of context

cited (6, first 6)
   {"cite": 1, "source": "models/apollo-11-sysml-v2/Technical/TechnicalComponentsPackage.sysml"}
   {"cite": 2, "source": "models/apollo-11-sysml-v2/Purpose/MissionPackage.sysml"}
   {"cite": 3, "source": "models/apollo-11-sysml-v2/Requirements/TechnicalRequirementsPackage.sysml"}
   {"cite": 4, "source": "models/DroneModelLogical.sysml"}
   {"cite": 5, "source": "models/apollo-11-sysml-v2/Technical/SystemSpecificationPackage.sysml"}
   {"cite": 6, "source": "models/apollo-11-sysml-v2/Analysis/CalculationsPackage.sysml"}

A  ## Answer

The model does not specify the cost to manufacture the F-1 engine alone. It provides some cost estimates connected to the Apollo 11 mission, such as the manufacturing cost being estimated at 3 billion dollars for the overall mission but does not break this estimate down to individual components such as the F-1 engine [CITE:2